# 🇧🇷 Fine-Tuning do BERTimbau para Detecção de Fake News

Neste notebook, implementamos o treinamento (*fine-tuning*) do **BERTimbau** (`neuralmind/bert-base-portuguese-cased`), um modelo Transformer pré-treinado no português brasileiro, para a tarefa de classificação binária de notícias:
- `0` = Notícia Falsa
- `1` = Notícia Verdadeira

### 🎯 Comparação com o Baseline
No notebook `treino.ipynb`, desenvolvemos a **Regressão Logística com TF-IDF** (nosso *baseline* honesto). Embora a acurácia tenha chegado a ~93%, a análise de atalhos mostrou que o modelo linear se apoiava em pistas superficiais (nomes de veículos de imprensa, palavras como "whatsapp" ou "vídeo", e o tamanho do texto).

Com o BERTimbau, o objetivo é permitir que a rede aprenda relações semânticas e contextuais mais profundas sobre o conteúdo real da notícia.

## 0. Verificação do Ambiente e Hardware (GPU / MPS)
Modelos Transformer exigem aceleração por hardware para treinar em tempo hábil. No macOS (Apple Silicon), utilizamos o backend **MPS** (*Metal Performance Shaders*) do PyTorch.

In [ ]:
import os
import torch

print(f"Versão do PyTorch: {torch.__version__}")

# Detecção automática de aceleração por hardware
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Dispositivo ativo: Apple Silicon GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Dispositivo ativo: GPU CUDA ({torch.cuda.get_device_name(0)})")
else:
    device = torch.device("cpu")
    print("⚠️ Dispositivo ativo: CPU (o treino pode ser mais lento)")

## 1. Carregamento dos Dados
Carregamos os dados que foram devidamente divididos e exportados ao final do `treino.ipynb`:
- `dados/export/treino.csv` (8.248 amostras de treino)
- `dados/export/validacao.csv` (1.768 amostras de validação)

> **Princípio de Honestidade Metodológica**: o conjunto de teste (`dados/export/teste.csv`) permanece intocado e lacrado até a validação final.

In [ ]:
import os
import pandas as pd

# Localizar caminhos dos dados com fallback para flexibilidade de execucao
candidatos_treino = ["dados/export/treino.csv", "../dados/export/treino.csv", "data/processed/treino.csv"]
candidatos_val = ["dados/export/validacao.csv", "../dados/export/validacao.csv", "data/processed/validacao.csv"]

path_treino = next((p for p in candidatos_treino if os.path.exists(p)), "dados/export/treino.csv")
path_val = next((p for p in candidatos_val if os.path.exists(p)), "dados/export/validacao.csv")

df_treino = pd.read_csv(path_treino)
df_val = pd.read_csv(path_val)

print(f"Treino ({path_treino}):    {len(df_treino)} noticias")
print(f"Validacao ({path_val}): {len(df_val)} noticias")
print()

# Verificar se ha valores ausentes (ML Best Practice)
print("Valores nulos no Treino:\n", df_treino.isna().sum())
print("Valores nulos na Validacao:\n", df_val.isna().sum())
print()

# Verificar o balanceamento das classes
print("Distribuicao de classes no Treino:")
print(df_treino["label"].value_counts().rename({0: "Falsa (0)", 1: "Verdadeira (1)"}))
print()

# Amostra visual
print("Exemplo da primeira noticia de treino:")
print(f"Rotulo: {"Verdadeira (1)" if df_treino["label"].iloc[0] == 1 else "Falsa (0)"}")
print(df_treino["texto"].iloc[0][:300], "...")

## 2. Tokenização com BERTimbau
O BERT não recebe texto puro diretamente. Ele utiliza um algoritmo chamado **WordPiece** que quebra o texto em subpalavras e as converte em números inteiros (IDs do vocabulário).

Tokens especiais incluídos:
- `[CLS]` (ID 101): colocado no início de toda sequência; o vetor desta posição é o resumo de toda a notícia e alimenta o classificador.
- `[SEP]` (ID 102): colocado no final de cada texto.
- `[PAD]` (ID 0): preenche sequências menores para uniformizar os lotes (*batches*).

In [ ]:
from transformers import AutoTokenizer

NOME_MODELO = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELO)

# Exemplo didático de tokenização em português
frase_teste = "Vacinas contra a COVID-19 são eficazes e seguras, afirma Anvisa."
tokens = tokenizer.tokenize(frase_teste)
ids = tokenizer.encode(frase_teste)

print("Texto original:       ", frase_teste)
print("Tokens de subpalavras:", tokens)
print("IDs correspondentes:  ", ids)
print("Tokens decodificados: ", tokenizer.convert_ids_to_tokens(ids))

## 3. Preparação dos Datasets Hugging Face
Convertemos os DataFrames para objetos `Dataset` e aplicamos a tokenização em lote (*batched*).
Fixamos o tamanho máximo em `256` tokens com truncamento. O `DataCollatorWithPadding` faz o preenchimento dinâmico eficiente por lote durante o treino.

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

# Converter para formato Hugging Face
dataset_treino = Dataset.from_pandas(df_treino)
dataset_val = Dataset.from_pandas(df_val)

def tokenizar(batch):
    return tokenizer(batch["texto"], truncation=True, max_length=256)

# Mapear a tokenização sobre todos os exemplos
dataset_treino = dataset_treino.map(tokenizar, batched=True)
dataset_val = dataset_val.map(tokenizar, batched=True)

# Colator para padding dinâmico (economiza memória de GPU)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Colunas do dataset pronto para o modelo:", dataset_treino.column_names)

## 4. Carregamento do Modelo Pré-treinado e Métricas
Instanciamos `AutoModelForSequenceClassification` com o BERTimbau.
Uma camada densa linear com 2 saídas é adicionada ao topo do BERT para a classificação binária.

### Métricas de Avaliação:
- **Acurácia**: porcentagem geral de acertos.
- **F1-Score**: balanço harmônico entre Precisão e Recall. Como nosso objetivo no challenge é não produzir classificadores enganosos, o F1 é a métrica principal para seleção do melhor modelo.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification

id2label = {0: "Falsa", 1: "Verdadeira"}
label2id = {"Falsa": 0, "Verdadeira": 1}

modelo = AutoModelForSequenceClassification.from_pretrained(
    NOME_MODELO,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

print("Modelo e métricas configurados!")

## 5. Configuração do Treinamento (`TrainingArguments` e `Trainer`)
Definimos os parâmetros essenciais:
- `learning_rate = 2e-5`: taxa de aprendizado baixa para preservar o conhecimento prévio da língua portuguesa.
- `num_train_epochs = 2`: 2 épocas são ideais para convergência rápida em um computador pessoal com MPS.
- `per_device_train_batch_size = 16`: tamanho de lote estável para memória compartilhada.
- `load_best_model_at_end = True`: ao final do treino, restaura automaticamente os pesos que alcançaram o melhor `f1` na validação.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./saida/bertimbau-fakenews",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    save_total_limit=1,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=dataset_treino,
    eval_dataset=dataset_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer inicializado com sucesso!")

## 6. Treinamento (*Fine-Tuning*)
Execute a célula abaixo para iniciar o treinamento do BERTimbau.
Acompanhe o decaimento da perda (`loss`) e as métricas calculadas na validação ao final de cada época.

In [ ]:
# Inicia o treinamento
resultado_treino = trainer.train()
print(resultado_treino)

## 7. Avaliação e Matriz de Confusão na Validação
Inspecionamos os erros do modelo: quantos falsos positivos (notícias falsas classificadas como verdadeiras) e falsos negativos ocorreram.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Fazer predições no conjunto de validação
predicoes_val = trainer.predict(dataset_val)
y_pred = np.argmax(predicoes_val.predictions, axis=1)
y_real = predicoes_val.label_ids

print("=== RELATÓRIO DE CLASSIFICAÇÃO (VALIDAÇÃO) ===")
print(classification_report(y_real, y_pred, target_names=["Falsa (0)", "Verdadeira (1)"]))

# Matriz de Confusão
cm = confusion_matrix(y_real, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Falsa (0)", "Verdadeira (1)"])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Matriz de Confusão - BERTimbau")
plt.show()

## 8. Teste Interativo em Tempo Real
Aqui você pode digitar ou colar qualquer notícia para testar a resposta do seu modelo treinado.

In [ ]:
import torch.nn.functional as F

def classificar_noticia(texto_noticia):
    modelo.eval()
    inputs = tokenizer(texto_noticia, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(modelo.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = modelo(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
        
    classe_predita = int(np.argmax(probs))
    rotulo = id2label[classe_predita]
    confianca = probs[classe_predita] * 100
    
    print("=" * 60)
    print(f"Texto: {texto_noticia[:160]}...")
    print(f"Predição: {rotulo} ({confianca:.1f}% de confiança)")
    print(f"Detalhes: Falsa={probs[0]*100:.1f}% | Verdadeira={probs[1]*100:.1f}%")
    print("=" * 60)
    return rotulo, probs

# Exemplo 1: Texto jornalístico formal
classificar_noticia("O Ministério da Educação divulgou nesta manhã as datas oficiais para as inscrições no programa de bolsas universitárias para o próximo semestre.")

# Exemplo 2: Texto sensacionalista com tom alarmista
classificar_noticia("URGENTE repasse para todos os seus grupos! O governo acaba de proibir remédios comuns nas farmácias e médicos alertam para plano secreto!")

## 9. Exportar o Modelo Treinado
Salvamos os pesos finais ajustados e o vocabulário do tokenizer para que você possa utilizá-los futuramente em uma API ou no app Lume.

In [ ]:
DIRETORIO_SALVAR = "./modelos/bertimbau-fakenews-final"
os.makedirs(DIRETORIO_SALVAR, exist_ok=True)

modelo.save_pretrained(DIRETORIO_SALVAR)
tokenizer.save_pretrained(DIRETORIO_SALVAR)

print(f"✅ Modelo e tokenizer salvos com sucesso em: {DIRETORIO_SALVAR}")

## 10. Conclusão e Próximos Passos

### Síntese dos Resultados:
1. **Comparação com o Baseline**: compare o F1-Score do BERTimbau com o F1-Score da Regressão Logística obtido no `treino.ipynb`.
2. **Resistência a Atalhos**: enquanto a Regressão Logística dependia fortemente da presença de veículos de comunicação ou tamanho do texto, o BERTimbau avalia a coerência semântica e estilística da afirmação inteira.
3. **Próxima Etapa no Challenge**: com o baseline e o transformer concluídos, o passo final é rodar a inferência no conjunto de teste lacrado (`dados/export/teste.csv`) para emitir o relatório comparativo definitivo do projeto.